# Mayoristas v3 -- punto de partida

`upt` (unidades por ticket) en v1/v2 suma `unidades` de TODOS los productos de un ticket sin importar `unidad_medida` -- mezcla KILOS, LITROS, Unid, Blis, Pack, Metros y Ramo como si fueran la misma cosa. Un cliente que compra 130 KILOS y 697 Unid en el mismo ticket queda con upt=827, que no representa ningun comportamiento real de compra.

Antes de rediseñar la metodologia, primero hay que ver la magnitud del problema:
1. Boxplot de unidades y venta por ticket, tal como lo consume v2 hoy (todas las unidades de medida mezcladas).
2. El mismo boxplot, desagregado por `unidad_medida` -- para ver si de verdad son escalas incompatibles.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns

sys.path.insert(0, str(Path.cwd()))

from analisis_mayoristas_v2 import ConfiguracionAnalisis
from conexion_bd import conexion_bd

plt.rcParams.update({
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "axes.grid.axis": "y",
    "grid.color": "#e1e0d9",
    "grid.linewidth": 0.6,
    "axes.edgecolor": "#c3c2b7",
    "text.color": "#0b0b0b",
    "axes.labelcolor": "#0b0b0b",
    "xtick.color": "#52514e",
    "ytick.color": "#52514e",
    "font.size": 11,
})

## Carga

Parte de `dbo.view_ventas_ahorrazo_filtradas_12m` (ya trae la ventana de 12 meses y las exclusiones de empresa/bolsa/egre-servi aplicadas) y le suma `unidad_medida` desde `dbo.Productos`, que la vista no expone.

In [ ]:
SQL_VENTAS_CON_UNIDAD = """
SELECT
    v.fecha_venta,
    v.pdv_id,
    v.ticked_id AS ticket_id,
    v.cliente_id,
    v.cliente_id_limpio,
    v.producto_id,
    p.unidad_medida,
    v.unidades,
    v.venta_gs
FROM dbo.view_ventas_ahorrazo_filtradas_12m v
INNER JOIN dbo.Productos p
    ON CAST(v.producto_id AS varchar(100)) = CAST(p.producto_id AS varchar(100))
"""

with conexion_bd() as conn:
    ventas_crudas = pd.read_sql_query(SQL_VENTAS_CON_UNIDAD, conn)

print(f"Filas cargadas: {len(ventas_crudas):,}")
ventas_crudas.head()

In [ ]:
# Mismas exclusiones que v2 (cliente de test + productos sin valor analitico) --
# reusa ConfiguracionAnalisis para no duplicar las listas.
config = ConfiguracionAnalisis()

ventas = ventas_crudas.copy()
ventas["fecha_venta"] = pd.to_datetime(ventas["fecha_venta"])
for col in ("cliente_id", "cliente_id_limpio", "producto_id", "pdv_id", "ticket_id"):
    ventas[col] = ventas[col].astype(str)

ventas = ventas[~ventas["cliente_id"].isin(config.excluir_clientes)]
ventas = ventas[~ventas["producto_id"].isin(config.excluir_productos)]
ventas["unidad_medida"] = ventas["unidad_medida"].fillna("SIN_UNIDAD")

print(f"Filas despues de limpieza: {len(ventas):,}")
print(f"Clientes unicos: {ventas['cliente_id_limpio'].nunique():,}")
print(f"Unidades de medida presentes: {sorted(ventas['unidad_medida'].unique())}")

## 1) Unidades y venta por ticket -- todas las unidades de medida mezcladas

Este es el insumo real de `upt` en v1/v2 hoy: suma de `unidades` por ticket sin discriminar que se esta sumando. Escala log en el eje Y -- la distribucion es muy asimetrica y sin log el boxplot queda ilegible.

In [ ]:
por_ticket = (
    ventas.groupby("ticket_id", dropna=False)
    .agg(unidades=("unidades", "sum"), venta_gs=("venta_gs", "sum"))
    .reset_index()
)
por_ticket = por_ticket[por_ticket["unidades"] > 0]

print(f"Tickets: {len(por_ticket):,}")
por_ticket[["unidades", "venta_gs"]].describe()

In [ ]:
caja_estilo = dict(
    patch_artist=True,
    boxprops=dict(facecolor="#cde2fb", edgecolor="#2a78d6"),
    medianprops=dict(color="#184f95", linewidth=2),
    whiskerprops=dict(color="#2a78d6"),
    capprops=dict(color="#2a78d6"),
    flierprops=dict(markeredgecolor="#2a78d6", markersize=3, alpha=0.4),
)

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].boxplot(por_ticket["unidades"], vert=True, widths=0.4, **caja_estilo)
axes[0].set_yscale("log")
axes[0].set_xticks([])
axes[0].set_ylabel("Unidades por ticket (escala log)")
axes[0].set_title("Unidades por ticket\n(todas las unidades de medida mezcladas)")

axes[1].boxplot(por_ticket["venta_gs"], vert=True, widths=0.4, **caja_estilo)
axes[1].set_yscale("log")
axes[1].set_xticks([])
axes[1].set_ylabel("Venta en Gs por ticket (escala log)")
axes[1].set_title("Venta por ticket")

plt.tight_layout()
plt.show()

## 2) Lo mismo, desagregado por `unidad_medida`

Grano cliente-ticket-unidad_medida -- si un ticket tiene lineas de KILOS y de Unid, aporta una fila a cada grupo. Un color fijo por unidad de medida (paleta categorica de `dataviz`, mismo orden en los dos paneles).

In [ ]:
por_ticket_unidad = (
    ventas.groupby(["ticket_id", "unidad_medida"], dropna=False)
    .agg(unidades=("unidades", "sum"), venta_gs=("venta_gs", "sum"))
    .reset_index()
)
por_ticket_unidad = por_ticket_unidad[por_ticket_unidad["unidades"] > 0]

orden_unidades = sorted(por_ticket_unidad["unidad_medida"].unique())
paleta_categorica = [
    "#2a78d6", "#eb6834", "#1baf7a", "#eda100",
    "#e87ba4", "#008300", "#4a3aa7", "#e34948",
]
paleta_por_unidad = dict(zip(orden_unidades, paleta_categorica))

por_ticket_unidad.groupby("unidad_medida")["unidades"].describe()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

sns.boxplot(
    data=por_ticket_unidad, x="unidad_medida", y="unidades", order=orden_unidades,
    hue="unidad_medida", palette=paleta_por_unidad, legend=False, width=0.5, ax=axes[0],
)
axes[0].set_yscale("log")
axes[0].set_xlabel("")
axes[0].set_ylabel("Unidades por ticket (escala log)")
axes[0].set_title("Unidades por ticket, por unidad de medida")
axes[0].tick_params(axis="x", rotation=30)

sns.boxplot(
    data=por_ticket_unidad, x="unidad_medida", y="venta_gs", order=orden_unidades,
    hue="unidad_medida", palette=paleta_por_unidad, legend=False, width=0.5, ax=axes[1],
)
axes[1].set_yscale("log")
axes[1].set_xlabel("")
axes[1].set_ylabel("Venta en Gs por ticket (escala log)")
axes[1].set_title("Venta por ticket, por unidad de medida")
axes[1].tick_params(axis="x", rotation=30)

plt.tight_layout()
plt.show()